# Week 2 · From NumPy to Gradient Descent
**Deep Learning & Reinforcement Learning (IE509)**

Last week we saw the history of neural networks. Every one of them learns the same way: it **measures how wrong it is, finds which direction makes it less wrong, and takes a small step that way.** That is *gradient descent*, and today we build it from scratch.

| Part | What we do |
|---|---|
| 1 | NumPy as a spreadsheet: arrays, shapes, the dot product |
| 2 | Plotting functions and data; fitting a line by hand |
| 3 | A function and its derivative: what the slope tells us |
| 4 | Gradient descent in one variable |
| 5 | Two variables: the gradient is a vector, and the dot product tells us the slope in any direction |
| 6 | Fitting a sales model with gradient descent |
| 7 | The same thing in PyTorch: the computer finds the slope for us |


**Cells with sliders are interactive: move the slider and watch the plot change.**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
from ipywidgets import interact, FloatSlider, IntSlider, Dropdown

In [ ]:
# Set a bunch of default parameters for plotting
plt.rcParams.update({"figure.figsize": (7, 4),
                     "axes.grid": True,
                     "grid.alpha": 0.3,
                     "font.size": 11,
                     "axes.spines.top": False,
                     "axes.spines.right": False})
BLUE, RED, GREEN, GREY, ORANGE = "#2563eb", "#dc2626", "#16a34a", "#6b7280", "#ea580c"
np.random.seed(0)
print("NumPy", np.__version__, "| PyTorch", torch.__version__)

## Part 1 · NumPy as a spreadsheet

A NumPy **array** is like a column (or a table) in Excel, but you work on the whole thing at once instead of cell by cell.

| Excel | NumPy | Name in maths |
|---|---|---|
| one cell | `3.5` | scalar |
| one column | `np.array([...])` | vector |
| a sheet (rows × columns) | 2-D array | matrix |
| several sheets | 3-D array | tensor |

You go shopping. Here are the prices and the quantities you buy:

In [ ]:
# bread, milk, cheese, eggs
prices = np.array([120, 80, 250, 40])
qty    = np.array([  2,  5,   1, 10])

print("prices:", prices, " shape:", prices.shape)
print("qty:   ", qty,    " shape:", qty.shape)

**Element-wise** operations work position by position, like dragging a formula down a column:

In [ ]:
cost_per_item = prices * qty
print("cost per item:", cost_per_item)
print("total bill:", cost_per_item.sum())

### The dot product = multiply pairwise, then add up

"Multiply pairwise and add up" is so common that it has its own name: the **dot product**. In Excel it is `SUMPRODUCT`.

$$\text{prices} \cdot \text{qty} = 120\cdot2 + 80\cdot5 + 250\cdot1 + 40\cdot10 = 1290$$

In NumPy you can write it with `np.dot` or with the `@` operator, like we saw in the other course:

In [ ]:
print(np.dot(prices, qty))
print(prices @ qty)

**Exercise 1.** Your final grade in this course is a weighted sum. The weights are attendance 10%, midterm 20%, homework 10%, final exam 60%. A student scored 90, 70, 100 and 65 (out of 100). Compute the final grade with a dot product.

In [ ]:
# Write the solution here
# print("final grade:", grade)

### A matrix: many baskets at once

Three customers, four products. Each **row** is one customer's basket. Multiplying the matrix by the price vector does one dot product per row and gives every customer's bill in one line.

The shape rule: `(3, 4) @ (4,)` → `(3,)`. The inner numbers (4 and 4) must match.

In [ ]:
baskets = np.array([[2, 5, 1, 10],    # customer A
                    [0, 2, 3,  6],    # customer B
                    [1, 1, 0, 12]])   # customer C
print("baskets shape:", baskets.shape)

prices = np.array([120, 80, 250, 40])

bills = baskets @ prices
print("bills:", bills, " shape:", bills.shape)

**Exercise 2.** Prices go up 10%. Compute everyone's new bill. Then try `baskets @ np.array([1, 2, 3])`: why does it fail? Read the error message.

In [ ]:
# Write the solution here
# print("new bills:", new_bills)

### Why this matters: a neuron is a dot product

Last week's perceptron computes $\;\hat y = \mathbf w \cdot \mathbf x + b\;$: inputs times weights, added up, plus a bias. That is a shopping bill where the **prices are the weights the network learns**.

In [ ]:
# inputs, e.g. three features of a customer
x = np.array([3.0, 1.5, 0.0])
# weights (the network learns these)
w = np.array([0.4, -1.0, 2.0])
# Add a bias term
b = 0.5

y_hat = w @ x + b
# 0.4*3 - 1.0*1.5 + 2.0*0 + 0.5 = 0.2
print("neuron output:", round(y_hat, 4))

## Part 2 · Plotting

To plot a function we evaluate it on many points and connect them. `np.linspace(a, b, n)` gives `n` evenly spaced points from `a` to `b`.

In [ ]:
x = np.linspace(-3, 3, 200)

plt.plot(x, x**2, color=BLUE, lw=2, label="$f(x) = x^2$")
plt.plot(x, 2*x + 1, color=ORANGE, lw=2, label="$g(x) = 2x + 1$")
plt.axhline(0, color="k", lw=0.8); plt.axvline(0, color="k", lw=0.8)
plt.legend(); plt.title("Two functions")
plt.show()

### Some data: advertising and sales

A firm records advertising spend (thousand €) and sales (thousand units) for 50 months. We want a line $\;\hat y = w\,x + b\;$ that predicts sales from advertising.

In [ ]:
rng = np.random.default_rng(42)
# advertising spend (random data): 50 points uniformly distributed from 0 to 10
ad    = rng.uniform(0, 10, 50)
# true slope 3, intercept 8, plus some noise (normally distributed)
sales = 3.0 * ad + 8.0 + rng.normal(0, 3, 50)

plt.scatter(ad, sales, color=BLUE, alpha=0.7)
plt.xlabel("advertising (thousand €)")
plt.ylabel("sales (thousand units)")
plt.title("50 months of data")
plt.show()

How good is a line? For each month we measure the error $(\hat y_i - y_i)$, square it so that negatives don't cancel positives, and average. This is the **mean squared error (MSE)**, our **loss**:

$$L(w, b) = \frac{1}{n}\sum_{i=1}^{n} (w\,x_i + b - y_i)^2$$

**Lower loss = better line.** Move the sliders and try to make the loss as small as you can.

In [ ]:
def mse(w, b):
    return np.mean((w * ad + b - sales) ** 2)

def plot_guess(w=1.0, b=0.0):
    xs = np.linspace(0, 10, 2)
    plt.scatter(ad, sales, color=BLUE, alpha=0.6)
    plt.plot(xs, w * xs + b, color=RED, lw=2)
    for xi, yi in zip(ad, sales):                     # draw the errors
        plt.plot([xi, xi], [yi, w * xi + b], color=GREY, lw=0.6, alpha=0.6)
    plt.ylim(-5, 50)
    plt.title(f"w = {w:.2f}, b = {b:.2f}   →   loss = {mse(w, b):.2f}")
    plt.xlabel("advertising"); plt.ylabel("sales"); plt.show()

interact(plot_guess, w=FloatSlider(value=1.0, min=-1, max=6, step=0.1),
                     b=FloatSlider(value=0.0, min=-10, max=25, step=0.5));

Guessing works for two numbers. A real network has **millions** of weights, so we need a method that finds the downhill direction automatically. For that we need the **derivative**.

## Part 3 · A function and its derivative

The **derivative** $f'(x)$ is the **slope** of $f$ at the point $x$: how much $f$ changes when $x$ moves a tiny bit.

We do not need calculus rules to compute it. We can just *measure* it: move $x$ a tiny step $h$ and see how much $f$ changes.

$$f'(x) \approx \frac{f(x+h) - f(x-h)}{2h}, \qquad h \text{ very small}$$

In [ ]:
def slope(f, x, h=1e-5):
    # numerical derivative: measure the change in f for a tiny change in x
    return (f(x + h) - f(x - h)) / (2 * h)

f = lambda x: x**2

print(" x   measured slope   rule 2x")
for x0 in [-2, -1, 0, 1, 2, 3]:
    print(f"{x0:2d}   {slope(f, x0):12.4f}   {2*x0:6d}")

The measured slope matches the textbook rule $f'(x) = 2x$.

### Plotting $f$ and $f'$ together

Top: the function. Bottom: its derivative. Read them together:

* where $f'(x) < 0$ (red zone), $f$ is going **down**;
* where $f'(x) > 0$ (green zone), $f$ is going **up**;
* where $f'(x) = 0$, $f$ is **flat**: a bottom (minimum) or a top (maximum).

In [ ]:
def plot_f_and_df(f, df, x, title, x0=None):
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(7, 6), sharex=True)
    d = df(x)
    for ax in (ax1, ax2):
        ax.fill_between(x, 0, 1, where=d < 0, color=RED,   alpha=0.07, transform=ax.get_xaxis_transform())
        ax.fill_between(x, 0, 1, where=d > 0, color=GREEN, alpha=0.07, transform=ax.get_xaxis_transform())
    ax1.plot(x, f(x), color=BLUE, lw=2)
    ax1.set_ylabel("$f(x)$"); ax1.set_title(title)
    ax2.plot(x, d, color=ORANGE, lw=2)
    ax2.axhline(0, color="k", lw=0.8)
    ax2.set_ylabel("$f'(x)$  (slope)"); ax2.set_xlabel("$x$")
    if x0 is not None:
        # tangent line on top: f(x0) + f'(x0) * (x - x0)
        ax1.plot(x, f(x0) + df(x0) * (x - x0), color=RED, lw=1.5, ls="--")
        ax1.plot(x0, f(x0), "o", color=RED, ms=8)
        ax1.set_ylim(f(x).min() - 1, f(x).max() + 1)
        ax2.plot(x0, df(x0), "o", color=RED, ms=8)
        ax2.vlines(x0, 0, df(x0), color=RED, lw=1.5)
    plt.tight_layout(); plt.show()

x = np.linspace(-3, 3, 400)
plot_f_and_df(lambda x: x**2, lambda x: 2*x, x, "$f(x) = x^2$ and its slope $f'(x) = 2x$")

Now a function with **two valleys**: $f(x) = x^4 - 3x^2 + x$. Its derivative crosses zero three times: two bottoms and one top.

In [ ]:
f2  = lambda x: x**4 - 3*x**2 + x
df2 = lambda x: 4*x**3 - 6*x + 1

x = np.linspace(-2.1, 2.1, 400)
plot_f_and_df(f2, df2, x, "$f(x) = x^4 - 3x^2 + x$: slope zero at every bottom and top")

### The slope at a point → the value nearby

Stand at a point $x_0$. The red dashed line is the **tangent**: the straight line with the same slope as $f$ at that point. It lets us **predict the value of $f$ nearby** using only two numbers, the value and the slope:

$$f(x_0 + \Delta) \;\approx\; f(x_0) + f'(x_0)\cdot\Delta$$

Move the slider. The red dot on the bottom plot is the slope at $x_0$. Its height equals the steepness of the dashed line on top.

In [ ]:
FUNCS = {"x²": (lambda x: x**2, lambda x: 2*x, np.linspace(-3, 3, 400)),
         "x⁴ − 3x² + x": (f2, df2, np.linspace(-2.1, 2.1, 400))}

def explore(function="x⁴ − 3x² + x", x0=1.5):
    f, df, x = FUNCS[function]
    x0 = float(np.clip(x0, x.min(), x.max()))
    plot_f_and_df(f, df, x, f"$x_0$ = {x0:.2f}   f($x_0$) = {f(x0):.2f}  slope f'($x_0$) = {df(x0):.2f}", x0=x0)

interact(explore, function=Dropdown(options=list(FUNCS)),
         x0=FloatSlider(value=1.5, min=-2.1, max=2.1, step=0.05));

How good is the tangent prediction? Very good for small steps, poor for big ones:

In [ ]:
x0 = 1.5
print(" step Δ   true f(x0+Δ)   predicted f(x0) + f'(x0)·Δ")
for delta in [0.01, 0.1, 0.5, 1.0]:
    true = f2(x0 + delta)
    pred = f2(x0) + df2(x0) * delta
    print(f"{delta:6.2f}   {true:12.4f}   {pred:12.4f}")

**Key lesson:** the slope tells us where to go, but we can only trust it for **small steps**. That is why gradient descent takes small steps.

**Exercise 3.** Use the `slope` function to measure the derivative of $g(x) = x^3$ at $x = 2$. The rule says $g'(x) = 3x^2$. Do they agree?

In [ ]:
# Write the solution here
# print("measured:", round(slope(g, 2.0), 4), "  rule:", 3 * 2.0**2)

## Part 4 · Gradient descent in one variable

You are on a hill in fog. You can't see the valley, but you can feel the slope under your feet. The rule:

* slope **positive** → downhill is to the **left** → move left;
* slope **negative** → downhill is to the **right** → move right.

In one formula: **step against the slope**.

$$x_{\text{new}} = x - \eta \cdot f'(x)$$

$\eta$ (eta) is the **learning rate**: how big a step we take.

In [ ]:
def gradient_descent_1d(df, x0, lr, steps):
    xs = [x0]
    x = x0
    for _ in range(steps):
        # the whole algorithm is this one line
        x = x - lr * df(x)
        xs.append(x)
    return np.array(xs)

path = gradient_descent_1d(df=lambda x: 2*x, x0=2.5, lr=0.1, steps=10)
print(np.round(path, 3))

**Exercise 4.** Without the function: start at $x = 3$ with learning rate 0.1 on $f(x) = x^2$ (slope $2x$). Do **three** steps by hand in code and print $x$ after each step.

In [ ]:
# write the solution here

### Watch it on both plots

Top: the walker on the function. Bottom: the slope at each step. As we approach the bottom, the slope shrinks, so the **steps shrink automatically**.

In [ ]:
def plot_descent_1d(f, df, x, path, title):
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(7, 6), sharex=True)
    ax1.plot(x, f(x), color=BLUE, lw=2)
    ax2.plot(x, df(x), color=ORANGE, lw=2); ax2.axhline(0, color="k", lw=0.8)
    visible = path[np.abs(path) < 1e3]
    colors = plt.cm.Reds(np.linspace(0.35, 1, len(visible)))
    ax1.plot(visible, f(visible), color=RED, lw=0.8, alpha=0.6)
    ax1.scatter(visible, f(visible), c=colors, s=40, zorder=3)
    ax2.scatter(visible, df(visible), c=colors, s=40, zorder=3)
    ax1.set_ylim(f(x).min() - 1, f(x).max() + 1); ax2.set_ylim(df(x).min() - 1, df(x).max() + 1)
    ax1.set_xlim(x.min(), x.max())
    ax1.set_ylabel("$f(x)$"); ax2.set_ylabel("$f'(x)$"); ax2.set_xlabel("$x$")
    ax1.set_title(title); plt.tight_layout(); plt.show()

x = np.linspace(-3, 3, 400)
path = gradient_descent_1d(lambda x: 2*x, x0=2.5, lr=0.1, steps=15)
plot_descent_1d(lambda x: x**2, lambda x: 2*x, x, path, "15 steps, learning rate 0.1 (dark = later steps)")

### The learning rate decides everything

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.8), sharey=True)
x = np.linspace(-3, 3, 400)
for ax, lr, name in zip(axes, [0.02, 0.3, 1.05], ["too small: creeping", "about right", "too big: diverges"]):
    p = gradient_descent_1d(lambda x: 2*x, x0=2.5, lr=lr, steps=12)
    p = p[np.abs(p) < 3]
    ax.plot(x, x**2, color=BLUE, lw=2)
    ax.plot(p, p**2, "o-", color=RED, ms=5, lw=0.8)
    ax.set_title(f"lr = {lr}: {name}"); ax.set_xlabel("$x$")
plt.tight_layout(); plt.show()

Try it yourself. Things to try:

1. On $x^2$, find the largest learning rate that still works. (Hint: something special happens at 1.0.)
2. On the two-valley function, start at $x_0 = 0.1$ and then at $x_0 = 0.3$. Which valley do you land in? (The hilltop between the valleys is at $x \approx 0.17$.)

In [ ]:
def explore_descent(function="x²", x0=2.5, lr=0.1, steps=15):
    f, df, x = FUNCS[function]
    path = gradient_descent_1d(df, x0, lr, steps)
    end = path[-1]
    msg = f"final x = {end:.3f}" if abs(end) < 1e3 else "DIVERGED!"
    plot_descent_1d(f, df, x, path, f"lr = {lr}, start = {x0}  →  {msg}")

interact(explore_descent, function=Dropdown(options=list(FUNCS)),
         x0=FloatSlider(value=2.0, min=-2.0, max=2.0, step=0.05),
         lr=FloatSlider(value=0.1, min=0.005, max=1.2, step=0.005, readout_format=".3f"),
         steps=IntSlider(value=15, min=1, max=60));

In [ ]:
# local minima: a tiny change in the starting point, a different valley
x = np.linspace(-2.1, 2.1, 400)
for start in [0.1, 0.3]:
    p = gradient_descent_1d(df2, x0=start, lr=0.02, steps=60)
    plot_descent_1d(f2, df2, x, p, f"start at {start}: ends at x = {p[-1]:.2f},  f = {f2(p[-1]):.2f}")

**Gradient descent finds *a* bottom, not necessarily *the* lowest one.** Where you start matters. This is one reason neural networks are trained with random starting weights, and sometimes several times.

## Part 5 · Two variables: the gradient is a vector

A model has many weights, so the loss depends on many variables. Start with two:

$$f(w_1, w_2) = w_1^2 + 3\,w_2^2$$

Now the "hill" is a surface. On the right is the same surface seen from above, like a contour map in hiking: each line connects points of equal height.

In [ ]:
def f_2d(w1, w2):
    return w1**2 + 3 * w2**2

W1, W2 = np.meshgrid(np.linspace(-3, 3, 100), np.linspace(-2, 2, 100))
Z = f_2d(W1, W2)

fig = plt.figure(figsize=(13, 4.5))
ax = fig.add_subplot(1, 2, 1, projection="3d")
ax.plot_surface(W1, W2, Z, cmap="viridis", alpha=0.9, linewidth=0)
ax.set_xlabel("$w_1$"); ax.set_ylabel("$w_2$"); ax.set_title("the surface $f(w_1, w_2)$")
ax2 = fig.add_subplot(1, 2, 2)
cs = ax2.contour(W1, W2, Z, levels=15, cmap="viridis")
ax2.clabel(cs, fontsize=8); ax2.set_aspect("equal")
ax2.set_xlabel("$w_1$"); ax2.set_ylabel("$w_2$"); ax2.set_title("the same surface from above (contours)")
plt.tight_layout(); plt.show()

With two variables there are **two slopes**: one if we move only in $w_1$, one if we move only in $w_2$. We measure each one exactly as before. Put them together and you get a vector, the **gradient**:

$$\nabla f = \begin{pmatrix} \text{slope in } w_1 \\ \text{slope in } w_2 \end{pmatrix} = \begin{pmatrix} 2w_1 \\ 6w_2 \end{pmatrix}$$

In [ ]:
def gradient_2d(f, w, h=1e-5):
    # measure the slope in each direction separately
    w = np.asarray(w, dtype=float)
    g = np.zeros_like(w)
    for i in range(len(w)):
        e = np.zeros_like(w); e[i] = h
        g[i] = (f(*(w + e)) - f(*(w - e))) / (2 * h)
    return g

w = np.array([1.0, 1.0])
print("measured gradient at (1, 1):", np.round(gradient_2d(f_2d, w), 4))
print("rule (2·w1, 6·w2):          ", np.array([2 * w[0], 6 * w[1]]))

### The gradient as arrows

At every point we draw the gradient as an arrow. Notice two things:

* the arrows point **uphill**, straight across the contour lines (at right angles to them);
* they are **longer where the surface is steeper**.

Gradient descent walks in the opposite direction, $-\nabla f$: **straight downhill**.

In [ ]:
G1, G2 = np.meshgrid(np.linspace(-2.7, 2.7, 13), np.linspace(-1.8, 1.8, 9))
U, V = 2 * G1, 6 * G2

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for ax, sign, color, title in [(axes[0], 1, RED, "gradient $\\nabla f$: points uphill"),
                               (axes[1], -1, GREEN, "minus gradient $-\\nabla f$: points downhill")]:
    ax.contour(W1, W2, Z, levels=15, cmap="viridis", alpha=0.6)
    ax.quiver(G1, G2, sign * U, sign * V, color=color, angles="xy", scale_units="xy", scale=25, width=0.004)
    ax.set_aspect("equal"); ax.set_title(title); ax.set_xlabel("$w_1$"); ax.set_ylabel("$w_2$")
plt.tight_layout(); plt.show()

### Where the dot product comes in (1): predicting the value nearby

In one variable: $f(x_0 + \Delta) \approx f(x_0) + f'(x_0)\cdot\Delta$.

In two variables we take a step $\boldsymbol\Delta = (\Delta_1, \Delta_2)$. The change is *slope in $w_1$ × step in $w_1$* **plus** *slope in $w_2$ × step in $w_2$*. Multiply pairwise and add up: that is a **dot product**, exactly the shopping bill from Part 1.

$$f(\mathbf w + \boldsymbol\Delta) \;\approx\; f(\mathbf w) + \nabla f \cdot \boldsymbol\Delta$$

In [ ]:
w = np.array([1.0, 1.0])
grad  = gradient_2d(f_2d, w)
# a small step
delta = np.array([0.10, -0.05])

true_value = f_2d(*(w + delta))
# value now + gradient · step
predicted  = f_2d(*w) + grad @ delta

print("gradient:", grad, "  step:", delta)
print("gradient · step =", round(grad @ delta, 4), " (= 2·0.10 + 6·(−0.05))")
print(f"true f(w + Δ) = {true_value:.4f}   predicted = {predicted:.4f}")

### Where the dot product comes in (2): the slope in *any* direction

Stand at $\mathbf w = (1, 1)$ and face some direction $\mathbf u$ (an arrow of length 1). The slope you feel in that direction is

$$\text{slope in direction } \mathbf u = \nabla f \cdot \mathbf u = \|\nabla f\| \cdot \cos(\text{angle between them})$$

The dot product is **largest when the two arrows point the same way** (cos = 1) and **most negative when they point opposite ways** (cos = −1). So:

* steepest **uphill** = along the gradient;
* steepest **downhill** = against the gradient. That's why gradient descent steps along $-\nabla f$.

In [ ]:
w = np.array([1.0, 1.0])
grad = gradient_2d(f_2d, w)
angles = np.linspace(0, 2 * np.pi, 361)
# 361 unit arrows, shape (361, 2)
directions = np.stack([np.cos(angles), np.sin(angles)], axis=1)
# one dot product per direction
slopes = directions @ grad

grad_angle = np.degrees(np.arctan2(grad[1], grad[0]))
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

ax = axes[0]
ax.contour(W1, W2, Z, levels=15, cmap="viridis", alpha=0.5)
for k in range(0, 360, 20):
    # arrows coloured by their slope
    d = directions[k]
    ax.arrow(*w, *(0.6 * d), color=plt.cm.coolwarm((slopes[k] / np.linalg.norm(grad) + 1) / 2),
             width=0.02, length_includes_head=True)
ax.arrow(*w, *(grad / np.linalg.norm(grad)), color="k", width=0.03, length_includes_head=True)
ax.annotate("gradient", w + grad / np.linalg.norm(grad) * 1.05, fontsize=10)
ax.set_xlim(-0.5, 2.5); ax.set_ylim(-0.5, 2.5); ax.set_aspect("equal")
ax.set_title("directions from (1, 1): red = uphill, blue = downhill")

ax = axes[1]
ax.plot(np.degrees(angles), slopes, color=BLUE, lw=2)
ax.axhline(0, color="k", lw=0.8)
ax.axvline(grad_angle, color=RED, ls="--", label=f"along gradient: slope = +{np.linalg.norm(grad):.2f}")
ax.axvline(grad_angle + 180, color=GREEN, ls="--", label=f"against gradient: slope = −{np.linalg.norm(grad):.2f}")
ax.set_xlabel("direction (degrees)"); ax.set_ylabel("slope = $\\nabla f \\cdot \\mathbf{u}$")
ax.set_title("slope in each direction"); ax.legend(fontsize=9)
plt.tight_layout(); plt.show()

The length of the gradient, $\|\nabla f\| = \sqrt{2^2 + 6^2} \approx 6.32$, is the steepest slope you can find at that point. (That length is called the **norm** of the vector.)

**Exercise 5.** At $\mathbf w = (1, 1)$, compute the slope when walking due east, $\mathbf u = (1, 0)$, and due north, $\mathbf u = (0, 1)$, using a dot product with `grad`. Which is steeper? Does it match the plot?

In [ ]:
# Write the solution here

### Gradient descent in 2D

Same rule as before, now with vectors: $\;\mathbf w_{\text{new}} = \mathbf w - \eta\,\nabla f(\mathbf w)$. Move the learning-rate slider. Around 0.3 the path starts to **zig-zag** across the narrow direction of the valley; above 1/3 it blows up.

In [ ]:
def gradient_descent_2d(grad_fn, w0, lr, steps):
    w = np.array(w0, dtype=float)
    path = [w.copy()]
    for _ in range(steps):
        w = w - lr * grad_fn(w)
        path.append(w.copy())
    return np.array(path)

grad_f2d = lambda w: np.array([2 * w[0], 6 * w[1]])

def explore_2d(lr=0.1, steps=25, start_w1=-2.5, start_w2=1.5):
    path = gradient_descent_2d(grad_f2d, [start_w1, start_w2], lr, steps)
    path = path[np.all(np.abs(path) < 10, axis=1)]
    plt.figure(figsize=(7, 4.5))
    plt.contour(W1, W2, Z, levels=15, cmap="viridis", alpha=0.6)
    plt.plot(path[:, 0], path[:, 1], "o-", color=RED, ms=4, lw=1)
    plt.plot(0, 0, "*", color="k", ms=14)
    plt.xlim(-3, 3); plt.ylim(-2, 2); plt.gca().set_aspect("equal")
    end = path[-1]
    plt.title(f"lr = {lr}: final point ({end[0]:.3f}, {end[1]:.3f}),  f = {f_2d(*end):.4f}")
    plt.xlabel("$w_1$"); plt.ylabel("$w_2$"); plt.show()

explore_2d(lr=0.1)
explore_2d(lr=0.3)
interact(explore_2d, lr=FloatSlider(value=0.1, min=0.01, max=0.36, step=0.01),
         steps=IntSlider(value=25, min=1, max=100),
         start_w1=FloatSlider(value=-2.5, min=-3, max=3, step=0.1),
         start_w2=FloatSlider(value=1.5, min=-2, max=2, step=0.1));

## Part 6 · Back to the sales data: learning $w$ and $b$

Now the hill is the **loss** from Part 2, and the two variables are the line's slope $w$ and intercept $b$.

The gradient of the MSE loss is (one line of calculus, which we'll trust and then check by measurement):

$$\frac{\partial L}{\partial w} = \frac{2}{n}\sum_i (\hat y_i - y_i)\,x_i = \frac{2}{n}\,(\hat{\mathbf y} - \mathbf y)\cdot\mathbf x
\qquad\qquad
\frac{\partial L}{\partial b} = \frac{2}{n}\sum_i (\hat y_i - y_i)$$

Look at the first one: **errors · inputs**, a dot product again. Months with large advertising and large errors push $w$ the hardest.

In [ ]:
def loss_and_grad(w, b):
    y_hat  = w * ad + b
    errors = y_hat - sales
    loss   = np.mean(errors ** 2)
    # dot product: errors · inputs
    dw     = 2 / len(ad) * (errors @ ad)
    db     = 2 / len(ad) * errors.sum()
    return loss, np.array([dw, db])

loss, g = loss_and_grad(1.0, 0.0)
print("loss at w=1, b=0:  ", round(loss, 2))
print("formula gradient:  ", np.round(g, 3))
print("measured gradient: ", np.round(gradient_2d(mse, [1.0, 0.0]), 3))

In [ ]:
def fit_line(lr=0.02, steps=1000, w0=0.0, b0=0.0):
    params = np.array([w0, b0])
    history, losses = [params.copy()], []
    for _ in range(steps):
        loss, g = loss_and_grad(*params)
        losses.append(loss)
        params = params - lr * g
        history.append(params.copy())
    return params, np.array(history), np.array(losses)

params, history, losses = fit_line(lr=0.02, steps=1000)
print(f"gradient descent:  w = {params[0]:.3f},  b = {params[1]:.3f},  loss = {losses[-1]:.3f}")
w_ls, b_ls = np.polyfit(ad, sales, 1)
print(f"exact least squares: w = {w_ls:.3f},  b = {b_ls:.3f}")

Three views of the same training run:

1. **The loss landscape** with the path of gradient descent.
2. **The line** improving over the steps.
3. **The loss curve**: loss against steps. You will see this curve every time you train a neural network.

In [ ]:
def plot_training(history, losses, lr):
    fig, axes = plt.subplots(1, 3, figsize=(15, 4.3))

    ws, bs = np.meshgrid(np.linspace(-0.5, 6.5, 120), np.linspace(-4, 16, 120))
    L = np.mean((ws[..., None] * ad + bs[..., None] - sales) ** 2, axis=-1)
    ax = axes[0]
    ax.contour(ws, bs, L, levels=np.geomspace(L.min() + 0.1, L.max(), 25), cmap="viridis", alpha=0.7)
    h = history[np.all(np.abs(history) < 100, axis=1)]
    ax.plot(h[:, 0], h[:, 1], ".-", color=RED, ms=3, lw=0.8)
    ax.plot(w_ls, b_ls, "*", color="k", ms=14)
    ax.set_xlim(-0.5, 6.5); ax.set_ylim(-4, 16)
    ax.set_xlabel("slope $w$"); ax.set_ylabel("intercept $b$"); ax.set_title(f"loss landscape, lr = {lr}")

    ax = axes[1]
    ax.scatter(ad, sales, color=BLUE, alpha=0.5, s=20)
    xs = np.array([0, 10])
    shown = [k for k in [0, 1, 3, 10, 50, len(history) - 1] if k < len(history)]
    for k, c in zip(shown, plt.cm.Reds(np.linspace(0.3, 1, len(shown)))):
        w, b = history[k]
        if abs(w) < 100 and abs(b) < 100:
            ax.plot(xs, w * xs + b, color=c, lw=2, label=f"step {k}")
    ax.set_ylim(-5, 50); ax.legend(fontsize=8); ax.set_title("the line improves")
    ax.set_xlabel("advertising"); ax.set_ylabel("sales")

    ax = axes[2]
    ax.plot(losses, color=BLUE, lw=2); ax.set_yscale("log")
    ax.set_xlabel("step"); ax.set_ylabel("loss (log scale)"); ax.set_title("loss curve")
    plt.tight_layout(); plt.show()

plot_training(history, losses, lr=0.02)

Notice the path: a fast jump into a **long narrow valley**, then a slow walk along it. This happens because $w$ and $b$ affect the loss on very different scales. (Later in the course: rescaling the inputs fixes this.)

Try different learning rates. What happens at 0.005? At 0.03?

In [ ]:
def explore_fit(lr=0.02, steps=300):
    _, h, l = fit_line(lr=lr, steps=steps)
    plot_training(h, l, lr)

interact(explore_fit, lr=FloatSlider(value=0.02, min=0.001, max=0.032, step=0.001, readout_format=".3f"),
         steps=IntSlider(value=300, min=10, max=2000, step=10));

### Many inputs: every prediction is a dot product

Real models have many inputs. Say sales depend on TV, radio and online advertising. Each month is a **row** $\mathbf x_i$ of a matrix $X$, and the prediction is a dot product with a weight vector:

$$\hat y_i = \mathbf w \cdot \mathbf x_i + b \qquad\Longrightarrow\qquad \hat{\mathbf y} = X\mathbf w + b \quad\text{(all months at once, like the baskets in Part 1)}$$

The gradient code barely changes: `errors @ X` does one dot product per weight.

In [ ]:
rng = np.random.default_rng(7)
n = 100
# columns: TV, radio, online
X = rng.uniform(0, 10, size=(n, 3))
true_w, true_b = np.array([2.0, 0.5, 3.5]), 4.0
y = X @ true_w + true_b + rng.normal(0, 2, n)
print("X shape:", X.shape, "  y shape:", y.shape)

w, b, lr = np.zeros(3), 0.0, 0.004
losses_multi = []
for step in range(20000):
    errors = X @ w + b - y
    losses_multi.append(np.mean(errors ** 2))
    w = w - lr * 2 / n * (errors @ X)
    b = b - lr * 2 / n * errors.sum()

print("learned weights:", np.round(w, 2), "  bias:", round(b, 2))
print("true weights:   ", true_w, "  bias:", true_b)
exact = np.linalg.lstsq(np.column_stack([X, np.ones(n)]), y, rcond=None)[0]
print("exact answer:   ", np.round(exact[:3], 2), "  bias:", round(exact[3], 2), " (gradient descent matches it; the gap to the true values is just noise in the data)")
plt.plot(losses_multi, color=BLUE, lw=2); plt.yscale("log")
plt.xlabel("step"); plt.ylabel("loss"); plt.title("3 inputs: loss curve"); plt.show()

## Part 7 · The same thing in PyTorch

In Part 6 we had to work out the gradient formula ourselves. With millions of weights and complicated networks, that's impossible by hand. **PyTorch computes gradients automatically** (this is called *autograd*).

First, a PyTorch **tensor** is almost the same as a NumPy array:

In [ ]:
t = torch.tensor([[1.0, 2.0], [3.0, 4.0]])
print(t, "\nshape:", t.shape)
print("t @ t:\n", t @ t)
print("back to NumPy:", t.numpy())

### Autograd: ask PyTorch for the slope

Mark a tensor with `requires_grad=True`, compute something with it, call `.backward()`, and the slope appears in `.grad`.

In [ ]:
x = torch.tensor(3.0, requires_grad=True)
f = x ** 2
f.backward()
print("f(3) =", f.item(), "   slope f'(3) from PyTorch =", x.grad.item(), "   rule 2x =", 2 * 3)

In [ ]:
# the two-variable function from Part 5
w = torch.tensor([1.0, 1.0], requires_grad=True)
f = w[0] ** 2 + 3 * w[1] ** 2
f.backward()
print("gradient from PyTorch:", w.grad, "   rule (2·w1, 6·w2) = (2, 6)")

### Fitting the sales line, PyTorch style

Compare with Part 6: **there is no gradient formula anywhere**. `loss.backward()` finds it.

In [ ]:
x_t = torch.tensor(ad, dtype=torch.float32)
y_t = torch.tensor(sales, dtype=torch.float32)

w = torch.tensor(0.0, requires_grad=True)
b = torch.tensor(0.0, requires_grad=True)
lr = 0.02

for step in range(1000):
    y_hat = w * x_t + b
    loss = ((y_hat - y_t) ** 2).mean()
    # PyTorch computes dloss/dw and dloss/db
    loss.backward()
    with torch.no_grad():
        # update the weights (don't track this step)
        w -= lr * w.grad
        b -= lr * b.grad
    # reset, otherwise gradients add up
    w.grad.zero_(); b.grad.zero_()

print(f"PyTorch:  w = {w.item():.3f}, b = {b.item():.3f}, loss = {loss.item():.3f}")
print(f"NumPy:    w = {params[0]:.3f}, b = {params[1]:.3f}")

### Even shorter: let an optimizer take the steps

`torch.optim.SGD` does the update and the reset for us. **This loop has the same shape as the training loop of every neural network in this course.** Only the model in the middle will get bigger.

In [ ]:
w = torch.tensor(0.0, requires_grad=True)
b = torch.tensor(0.0, requires_grad=True)
optimizer = torch.optim.SGD([w, b], lr=0.02)

losses_torch = []
for step in range(1000):
    # 1. reset gradients
    optimizer.zero_grad()
    # 2. predict and measure the error
    loss = ((w * x_t + b - y_t) ** 2).mean()
    # 3. compute gradients
    loss.backward()
    # 4. step downhill
    optimizer.step()
    losses_torch.append(loss.item())

print(f"w = {w.item():.3f}, b = {b.item():.3f}")
plt.plot(losses_torch, color=BLUE, lw=2); plt.yscale("log")
plt.xlabel("step"); plt.ylabel("loss"); plt.title("training with torch.optim.SGD"); plt.show()

---
## Summary

| Idea | What it means | Where you'll see it again |
|---|---|---|
| Dot product | multiply pairwise, add up | every neuron: $\mathbf w\cdot\mathbf x + b$ |
| Derivative | slope of a function at a point | the direction to improve |
| Gradient | the slopes for all weights, as a vector; points uphill | every training step |
| $\nabla f \cdot \boldsymbol\Delta$ | how much $f$ changes for a small step $\boldsymbol\Delta$ | why small steps are safe |
| Gradient descent | $\mathbf w \leftarrow \mathbf w - \eta\,\nabla L$ | training every network in this course |
| Learning rate | step size: too small = slow, too big = diverges | the first thing you tune |
| Autograd | PyTorch computes gradients for us | from week 5 onward |

### Exit questions
1. Your loss curve jumps up and down and gets bigger. What do you change?
2. The gradient at some point is $(4, -3)$. In which direction does gradient descent step?
3. In $\hat y = \mathbf w \cdot \mathbf x + b$, which numbers does the network learn, and which come from the data?

### Homework
Take the sales example (Part 6) and run gradient descent with **three different learning rates** of your choice. For each, save the loss curve plot and write 3–4 sentences: did it converge, how fast, and why? Submit the notebook.